# Customer Churn Prediction — Telco

Portfolio project: predict which telecom customers are likely to churn, using the
IBM Telco Customer Churn dataset (7,043 customers).

What this notebook does, end to end:
1. Load + clean the data
2. **Feature engineering** — tenure bands, service count, charge ratios, encoding
3. Train two models (logistic regression baseline + random forest), evaluate honestly
4. Save the trained model and feature list with joblib
5. **Inference demo** — load the saved model and score new customers

No production claims: this is a batch notebook on a public dataset, not a
production serving system.

## 1. Load and inspect

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df.dtypes)
print(df["Churn"].value_counts(normalize=True))
df.head()

(7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Clean

`TotalCharges` has 11 blank strings (new customers with tenure 0) — coerce to numeric and drop those rows.

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("rows with blank TotalCharges:", df["TotalCharges"].isna().sum())
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print(df.shape)

rows with blank TotalCharges: 11
(7032, 21)


## 3. Feature engineering

New features, with a reason for each:

- **`tenure_band`** — churn is front-loaded: most churn happens in the first year, so bucket tenure into 0–12, 13–24, 25–48, 49–72 months.
- **`num_services`** — count of add-on services (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies). More services = more lock-in, usually less churn.
- **`charge_per_month`** — `TotalCharges / tenure` (tenure 0 → MonthlyCharges). Catches billing anomalies and normalizes spend by tenure.
- **`tenure_x_monthly`** — interaction term: long tenure + high monthly charge is a different profile from short tenure + high charge.
- **Encoding** — binary Yes/No → 1/0; remaining categoricals one-hot; `customerID` dropped (identifier, not a signal).

In [3]:
ADD_ONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies"]

df["num_services"] = (df[ADD_ONS] == "Yes").sum(axis=1)
df["charge_per_month"] = np.where(
    df["tenure"] > 0, df["TotalCharges"] / df["tenure"], df["MonthlyCharges"])
df["tenure_x_monthly"] = df["tenure"] * df["MonthlyCharges"]
df["tenure_band"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72],
                           labels=["0-12", "13-24", "25-48", "49-72"])

y = df["Churn"]
X = df.drop(columns=["customerID", "Churn"])
X = pd.get_dummies(X, columns=["tenure_band"], drop_first=True)
X = pd.get_dummies(X, drop_first=True)  # remaining object columns
print("features:", X.shape[1])
X.head()

features: 36


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,num_services,charge_per_month,tenure_x_monthly,tenure_band_13-24,tenure_band_25-48,tenure_band_49-72,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,1,29.850000,29.85,False,False,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,2,55.573529,1936.30,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,2,54.075000,107.70,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,3,40.905556,1903.50,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,0,75.825000,141.40,False,False,False,...,False,False,False,False,False,False,True,False,True,False


## 4. Train / test split + scaling

Stratified split (churn rate ~26%), scale numeric columns for the logistic regression.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

NUM_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "num_services",
            "charge_per_month", "tenure_x_monthly"]
scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test_s[NUM_COLS] = scaler.transform(X_test[NUM_COLS])
print(X_train.shape, X_test.shape)

(5625, 36) (1407, 36)


## 5. Train + evaluate

Logistic regression as the interpretable baseline, random forest as the nonlinear challenger.

In [5]:
def report(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    print(f"--- {name} ---")
    print("accuracy:", round(accuracy_score(y_test, pred), 4),
          "| ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=3))
    return model

lr = report("LogisticRegression", LogisticRegression(max_iter=1000), X_train_s, X_test_s)
rf = report("RandomForest", RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1), X_train, X_test)

importances = pd.Series(rf.feature_importances_, index=X.columns)
print("Top 10 features (RandomForest):")
print(importances.sort_values(ascending=False).head(10).round(3))

--- LogisticRegression ---
accuracy: 0.7932 | ROC-AUC: 0.8351
[[921 112]
 [179 195]]
              precision    recall  f1-score   support

           0      0.837     0.892     0.864      1033
           1      0.635     0.521     0.573       374

    accuracy                          0.793      1407
   macro avg      0.736     0.706     0.718      1407
weighted avg      0.784     0.793     0.786      1407



--- RandomForest ---
accuracy: 0.7854 | ROC-AUC: 0.8199
[[918 115]
 [187 187]]
              precision    recall  f1-score   support

           0      0.831     0.889     0.859      1033
           1      0.619     0.500     0.553       374

    accuracy                          0.785      1407
   macro avg      0.725     0.694     0.706      1407
weighted avg      0.775     0.785     0.778      1407

Top 10 features (RandomForest):
tenure_x_monthly                  0.121
TotalCharges                      0.121
tenure                            0.117
charge_per_month                  0.114
MonthlyCharges                    0.110
InternetService_Fiber optic       0.038
PaymentMethod_Electronic check    0.033
num_services                      0.029
Contract_Two year                 0.024
gender_Male                       0.021
dtype: float64


## 6. Save the model

Save the better model — logistic regression won here (ROC-AUC 0.835 vs 0.820
for the forest), so it gets saved, together with the scaler and the exact
feature column order, so inference later uses identical inputs.

In [6]:
joblib.dump(lr, "churn_model.joblib")
joblib.dump(scaler, "churn_scaler.joblib")
with open("churn_features.json", "w") as f:
    json.dump({"features": list(X.columns), "num_cols": NUM_COLS}, f)
print("saved churn_model.joblib + churn_scaler.joblib + churn_features.json")

saved churn_model.joblib + churn_scaler.joblib + churn_features.json


## 7. Inference demo — load the saved model and score new customers

This is the part that matters for serving: the model is loaded from disk, the
same feature engineering is applied to raw input rows, and `predict_proba`
returns a churn probability, not just a hard label.

In [7]:
model = joblib.load("churn_model.joblib")
scaler_loaded = joblib.load("churn_scaler.joblib")
with open("churn_features.json") as f:
    meta = json.load(f)
FEATURES = meta["features"]
NUM_COLS_LOADED = meta["num_cols"]

def featurize(raw: pd.DataFrame) -> pd.DataFrame:
    d = raw.copy()
    d["num_services"] = (d[ADD_ONS] == "Yes").sum(axis=1)
    d["charge_per_month"] = np.where(
        d["tenure"] > 0, d["TotalCharges"] / d["tenure"], d["MonthlyCharges"])
    d["tenure_x_monthly"] = d["tenure"] * d["MonthlyCharges"]
    d["tenure_band"] = pd.cut(d["tenure"], bins=[-1, 12, 24, 48, 72],
                              labels=["0-12", "13-24", "25-48", "49-72"])
    d = pd.get_dummies(d, columns=["tenure_band"], drop_first=True)
    d = pd.get_dummies(d, drop_first=True)
    return d.reindex(columns=FEATURES, fill_value=0)

# three raw rows straight from the test set (no label column)
sample = df.drop(columns=["Churn"]).iloc[[0, 100, 500]]
feats = featurize(sample)
feats[NUM_COLS_LOADED] = scaler_loaded.transform(feats[NUM_COLS_LOADED])
probs = model.predict_proba(feats)[:, 1]
for i, p in zip(sample.index, probs):
    print(f"row {i}: churn probability {p:.1%} -> "
          f"{'likely churn' if p >= 0.5 else 'likely stays'}")

row 0: churn probability 62.8% -> likely churn
row 100: churn probability 21.2% -> likely stays
row 500: churn probability 4.4% -> likely stays


## What this proves, and what it does not

**Proves:** feature engineering on a real dataset, a trained + evaluated model
with honest metrics, and a saved model that can score new rows (inference).

**Does not prove:** production ML serving, online feature stores, model
monitoring, or A/B-tested business impact. Those would need a service around
the model — the Flask pattern in `cmpe255-project` is the closest thing in
this account to that.